## ℹ️ Sobre este Notebook

Este notebook demonstra **operações de limpeza e preparação de dados** em PySpark DataFrames usando dados de Bitcoin. Inclui exemplos de remoção de valores nulos (`drop`, `fillna`), imputação com `Imputer`, e análise de correlação entre preços e volume de negociação.


# First steps com dados BTC
* Different options to drop
* Fill nan
* Imputer
* Correlacao entre precos e volume


In [1]:
import pyspark

In [2]:
# Import PySpark and create a SparkSession
from pyspark.sql import SparkSession
# https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.html#pyspark.sql.DataFrameReader
spark = SparkSession.builder.appName("ImportData").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/18 21:42:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark

# Ler o ficheiro BTC (Parquet)


In [4]:
# Ler dados BTC em formato Parquet (dados reais da Binance)
# https://spark.apache.org/docs/latest/sql-data-sources-parquet.html
df_btc = spark.read.parquet('./../data/btc_04h_usdt_binance.parquet')

In [5]:
df_btc.show(10, truncate=30)


26/04/18 21:42:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------------------+-------+-------+-------+-------+----------+-----------------------+----------------+------+--------------+---------------+------+-------+-------------------+-------------------+----------+---------+---------------------+-------------+-------------------+-----------+----------+------------------+--------------+--------------+--------------+--------------+--------------+---------------+---------------+--------------+------------------+------------------+----------------+--------------------+---------------+---------------+--------------+--------------+--------------+--------------+--------------+-----------------+-------------+----------+-----------------+---------------+--------------+--------------+--------------+--------------+--------------------+--------------------+---------------------+----------+----------------+---------+---------+-------------+--------------+-------------------+-------------------+----------------+------------------+---------+---------+----

In [6]:
type(df_btc)


pyspark.sql.classic.dataframe.DataFrame

In [7]:
df_btc.describe().show(truncate=10)


26/04/18 21:42:08 WARN DAGScheduler: Broadcasting large task binary with size 1345.3 KiB


+-------+----------+----------+----------+----------+----------+----------+----------+--------------+---------------+------+-------+----------+----------+----------+----------+----------+-------------+----------+-----------+----------+----------+--------------+--------------+--------------+--------------+--------------+---------------+---------------+--------------+--------------+--------------+--------------+--------------+---------------+---------------+--------------+--------------+--------------+--------------+--------------+--------------+-------------+----------+-----------------+---------------+--------------+--------------+--------------+--------------+--------------------+--------------------+---------------------+----------+----------------+----------+----------+-------------+--------------+-------------------+-------------------+----------------+----------------+----------+----------+-------------+-------------+----------+-----------------------+-----------------------+-----

In [8]:
df_btc.columns


['open_time',
 'open',
 'high',
 'low',
 'close',
 'volume',
 'close_time',
 'quote_vol',
 'trades',
 'taker_buy_base',
 'taker_buy_quote',
 'ignore',
 'symbol',
 'volume_adi',
 'volume_obv',
 'volume_cmf',
 'volume_fi',
 'volume_em',
 'volume_sma_em',
 'volume_vpt',
 'volume_vwap',
 'volume_mfi',
 'volume_nvi',
 'volatility_bbm',
 'volatility_bbh',
 'volatility_bbl',
 'volatility_bbw',
 'volatility_bbp',
 'volatility_bbhi',
 'volatility_bbli',
 'volatility_kcc',
 'volatility_kch',
 'volatility_kcl',
 'volatility_kcw',
 'volatility_kcp',
 'volatility_kchi',
 'volatility_kcli',
 'volatility_dcl',
 'volatility_dch',
 'volatility_dcm',
 'volatility_dcw',
 'volatility_dcp',
 'volatility_atr',
 'volatility_ui',
 'trend_macd',
 'trend_macd_signal',
 'trend_macd_diff',
 'trend_sma_fast',
 'trend_sma_slow',
 'trend_ema_fast',
 'trend_ema_slow',
 'trend_vortex_ind_pos',
 'trend_vortex_ind_neg',
 'trend_vortex_ind_diff',
 'trend_trix',
 'trend_mass_index',
 'trend_dpo',
 'trend_kst',
 'trend_kst

In [9]:
len(df_btc.columns)


112

In [10]:
features = ['open', 'high', 'low', 'close', 'volume']


In [11]:
df_btc[features].show()


+-------+-------+-------+-------+----------+
|   open|   high|    low|  close|    volume|
+-------+-------+-------+-------+----------+
|4261.48|4349.99|4261.32|4349.99| 82.088865|
|4333.32|4485.39|4333.32| 4427.3| 63.619882|
|4436.06|4485.39|4333.42|4352.34|174.562001|
|4352.33|4354.84|4200.74|4325.23|225.109716|
|4307.56|4369.69|4258.56|4285.08|249.769913|
|4285.08|4340.62|4134.61|4292.39|276.193043|
|4292.39|4340.62|4234.43|4300.25|248.389029|
| 4285.0|4371.52|4259.85|4340.31|196.140129|
|4320.52|4340.31| 4193.7|4236.89|175.969384|
|4234.54|4297.75| 4015.7|4136.28| 193.64697|
| 4135.0|4136.48|3938.77|4108.37|109.549709|
|4108.37|4184.69|4084.28|4138.55|108.510448|
| 4138.5| 4156.0|3933.21|4033.47|112.752079|
|4033.47|4082.25| 3850.0| 3957.6| 77.256042|
|3945.12| 4100.0|3928.89|3972.05| 47.515578|
|4036.19|4103.92| 3953.4|4076.12| 25.925056|
|4088.71|4149.99|4051.47|4139.98|   9.35056|
|4120.98|4139.98| 4044.0|4094.62|  6.611283|
|4094.62|4156.56|4081.19|4155.87|  28.60391|
|4155.87|4

In [12]:
df_btc.select(features).show(10, truncate=30)


+-------+-------+-------+-------+----------+
|   open|   high|    low|  close|    volume|
+-------+-------+-------+-------+----------+
|4261.48|4349.99|4261.32|4349.99| 82.088865|
|4333.32|4485.39|4333.32| 4427.3| 63.619882|
|4436.06|4485.39|4333.42|4352.34|174.562001|
|4352.33|4354.84|4200.74|4325.23|225.109716|
|4307.56|4369.69|4258.56|4285.08|249.769913|
|4285.08|4340.62|4134.61|4292.39|276.193043|
|4292.39|4340.62|4234.43|4300.25|248.389029|
| 4285.0|4371.52|4259.85|4340.31|196.140129|
|4320.52|4340.31| 4193.7|4236.89|175.969384|
|4234.54|4297.75| 4015.7|4136.28| 193.64697|
+-------+-------+-------+-------+----------+
only showing top 10 rows


# Drop e tratamento de nulls
* Vamos utilizar isnull
* Aplicar ao dataset BTC


In [13]:
from pyspark.sql.functions import col,isnan,isnull,when

# Verificar nulls na coluna volume
df_btc.select("open_time", "close", "volume").where(isnull("volume")).show()

+---------+-----+------+
|open_time|close|volume|
+---------+-----+------+
+---------+-----+------+



In [14]:
df_btc.na.fill(0).show(truncate=20)


+-------------------+-------+-------+-------+-------+----------+--------------------+----------------+------+--------------+---------------+------+-------+-------------------+-------------------+-------------------+-------------------+--------------------+--------------------+--------------------+-----------------+------------------+------------------+--------------+-----------------+------------------+------------------+------------------+---------------+---------------+------------------+------------------+------------------+------------------+--------------------+---------------+---------------+--------------+--------------+--------------+----------------+-----------------+------------------+-----------------+----------+-----------------+---------------+------------------+--------------+-----------------+--------------+--------------------+--------------------+---------------------+----------+----------------+-----------------+------------------+------------------+------------------

In [15]:
df_btc.na.drop().show(truncate=20)


+---------+----+----+---+-----+------+----------+---------+------+--------------+---------------+------+------+----------+----------+----------+---------+---------+-------------+----------+-----------+----------+----------+--------------+--------------+--------------+--------------+--------------+---------------+---------------+--------------+--------------+--------------+--------------+--------------+---------------+---------------+--------------+--------------+--------------+--------------+--------------+--------------+-------------+----------+-----------------+---------------+--------------+--------------+--------------+--------------+--------------------+--------------------+---------------------+----------+----------------+---------+---------+-------------+--------------+-------------------+-------------------+----------------+----------------+---------+---------+-------------+-------------+---------+-----------------------+-----------------------+--------------+----------------+-

In [16]:
# drop any row with null in volume
df_btc.na.drop(how="any", subset=['volume']).show(truncate=5)


+---------+-----+-----+-----+-----+------+----------+---------+------+--------------+---------------+------+------+----------+----------+----------+---------+---------+-------------+----------+-----------+----------+----------+--------------+--------------+--------------+--------------+--------------+---------------+---------------+--------------+--------------+--------------+--------------+--------------+---------------+---------------+--------------+--------------+--------------+--------------+--------------+--------------+-------------+----------+-----------------+---------------+--------------+--------------+--------------+--------------+--------------------+--------------------+---------------------+----------+----------------+---------+---------+-------------+--------------+-------------------+-------------------+----------------+----------------+---------+---------+-------------+-------------+---------+-----------------------+-----------------------+--------------+--------------

In [17]:
# fill volume nulls with 0
df_btc.na.fill(0, ["volume"]).show(truncate=5)


+---------+-----+-----+-----+-----+------+----------+---------+------+--------------+---------------+------+------+----------+----------+----------+---------+---------+-------------+----------+-----------+----------+----------+--------------+--------------+--------------+--------------+--------------+---------------+---------------+--------------+--------------+--------------+--------------+--------------+---------------+---------------+--------------+--------------+--------------+--------------+--------------+--------------+-------------+----------+-----------------+---------------+--------------+--------------+--------------+--------------+--------------------+--------------------+---------------------+----------+----------------+---------+---------+-------------+--------------+-------------------+-------------------+----------------+----------------+---------+---------+-------------+-------------+---------+-----------------------+-----------------------+--------------+--------------

# Imputer
* Preenche valores em falta com a mediana


In [18]:
from pyspark.ml.feature import Imputer

imputer = Imputer(
    inputCols=['volume'], 
    outputCols=["{}_imputed".format(c) for c in ['volume']]
    ).setStrategy("median")


In [19]:
imputer.fit(df_btc).transform(df_btc).show(truncate=5)


+---------+-----+-----+-----+-----+------+----------+---------+------+--------------+---------------+------+------+----------+----------+----------+---------+---------+-------------+----------+-----------+----------+----------+--------------+--------------+--------------+--------------+--------------+---------------+---------------+--------------+--------------+--------------+--------------+--------------+---------------+---------------+--------------+--------------+--------------+--------------+--------------+--------------+-------------+----------+-----------------+---------------+--------------+--------------+--------------+--------------+--------------------+--------------------+---------------------+----------+----------------+---------+---------+-------------+--------------+-------------------+-------------------+----------------+----------------+---------+---------+-------------+-------------+---------+-----------------------+-----------------------+--------------+--------------

# Matriz de correlacao BTC
* Analisar relacoes entre OHLCV
* Converter para vector com 1 dimensao
* Visualizar com heatmap


In [20]:
# Ler dados BTC em formato Parquet (dados reais da Binance)
# https://spark.apache.org/docs/latest/sql-data-sources-parquet.html
df_btc = spark.read.parquet('./../data/btc_04h_usdt_binance.parquet')

In [21]:
from pyspark.ml.stat import Correlation
from pyspark.ml.linalg import Vectors
from pyspark.ml.feature import VectorAssembler
import pyspark.sql.functions as F

In [22]:
df_btc[features].show(5)


+-------+-------+-------+-------+----------+
|   open|   high|    low|  close|    volume|
+-------+-------+-------+-------+----------+
|4261.48|4349.99|4261.32|4349.99| 82.088865|
|4333.32|4485.39|4333.32| 4427.3| 63.619882|
|4436.06|4485.39|4333.42|4352.34|174.562001|
|4352.33|4354.84|4200.74|4325.23|225.109716|
|4307.56|4369.69|4258.56|4285.08|249.769913|
+-------+-------+-------+-------+----------+
only showing top 5 rows


## Verificar valores nulos
```python
df_btc.select([F.count(F.when(F.isnan(c) | F.col(c).isNull(), c)).alias(c) for c in features]).show()
```
* Conta quantos nulls existem em cada coluna numerica


In [23]:
import pyspark.sql.functions as F

# Verificar nulls em todas as features numericas
df_btc.select([F.count(F.when(F.isnan(c) | F.col(c).isNull(), c)).alias(c) for c in features]).show()


+----+----+---+-----+------+
|open|high|low|close|volume|
+----+----+---+-----+------+
|   0|   0|  0|    0|     0|
+----+----+---+-----+------+



In [24]:
df_btc.filter(df_btc['volume'].isNull()).show(5)


+---------+----+----+---+-----+------+----------+---------+------+--------------+---------------+------+------+----------+----------+----------+---------+---------+-------------+----------+-----------+----------+----------+--------------+--------------+--------------+--------------+--------------+---------------+---------------+--------------+--------------+--------------+--------------+--------------+---------------+---------------+--------------+--------------+--------------+--------------+--------------+--------------+-------------+----------+-----------------+---------------+--------------+--------------+--------------+--------------+--------------------+--------------------+---------------------+----------+----------------+---------+---------+-------------+--------------+-------------------+-------------------+----------------+----------------+---------+---------+-------------+-------------+---------+-----------------------+-----------------------+--------------+----------------+-

In [25]:
df_btc.filter(df_btc['close'].isNull()).show(10)


+---------+----+----+---+-----+------+----------+---------+------+--------------+---------------+------+------+----------+----------+----------+---------+---------+-------------+----------+-----------+----------+----------+--------------+--------------+--------------+--------------+--------------+---------------+---------------+--------------+--------------+--------------+--------------+--------------+---------------+---------------+--------------+--------------+--------------+--------------+--------------+--------------+-------------+----------+-----------------+---------------+--------------+--------------+--------------+--------------+--------------------+--------------------+---------------------+----------+----------------+---------+---------+-------------+--------------+-------------------+-------------------+----------------+----------------+---------+---------+-------------+-------------+---------+-----------------------+-----------------------+--------------+----------------+-

In [26]:
# Remover nulls em volume e close
df_btc = df_btc.dropna(subset=['volume', 'close'])


In [27]:
import pyspark.sql.functions as F

# Verificar nulls em todas as features numericas
df_btc.select([F.count(F.when(F.isnan(c) | F.col(c).isNull(), c)).alias(c) for c in features]).show()


+----+----+---+-----+------+
|open|high|low|close|volume|
+----+----+---+-----+------+
|   0|   0|  0|    0|     0|
+----+----+---+-----+------+



In [28]:
df_btc.show(5)


+-------------------+-------+-------+-------+-------+----------+--------------------+----------------+------+--------------+---------------+------+-------+-------------------+------------------+----------+---------+--------------------+-------------+-------------------+-----------+----------+------------------+--------------+--------------+--------------+--------------+--------------+---------------+---------------+--------------+------------------+-----------------+--------------+-------------------+---------------+---------------+--------------+--------------+--------------+--------------+--------------+--------------+-------------+----------+-----------------+---------------+--------------+--------------+--------------+--------------+--------------------+--------------------+---------------------+----------+----------------+---------+---------+-------------+--------------+-------------------+-------------------+----------------+------------------+---------+---------+-------------+--

In [29]:
features

['open', 'high', 'low', 'close', 'volume']

In [30]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

assembler = VectorAssembler(inputCols=features, outputCol="features")
df_assembled = assembler.transform(df_btc)
df_assembled.select("open_time", "close", "features").show()

+-------------------+-------+--------------------+
|          open_time|  close|            features|
+-------------------+-------+--------------------+
|2017-08-17 04:00:00|4349.99|[4261.48,4349.99,...|
|2017-08-17 08:00:00| 4427.3|[4333.32,4485.39,...|
|2017-08-17 12:00:00|4352.34|[4436.06,4485.39,...|
|2017-08-17 16:00:00|4325.23|[4352.33,4354.84,...|
|2017-08-17 20:00:00|4285.08|[4307.56,4369.69,...|
|2017-08-18 00:00:00|4292.39|[4285.08,4340.62,...|
|2017-08-18 04:00:00|4300.25|[4292.39,4340.62,...|
|2017-08-18 08:00:00|4340.31|[4285.0,4371.52,4...|
|2017-08-18 12:00:00|4236.89|[4320.52,4340.31,...|
|2017-08-18 16:00:00|4136.28|[4234.54,4297.75,...|
|2017-08-18 20:00:00|4108.37|[4135.0,4136.48,3...|
|2017-08-19 00:00:00|4138.55|[4108.37,4184.69,...|
|2017-08-19 04:00:00|4033.47|[4138.5,4156.0,39...|
|2017-08-19 08:00:00| 3957.6|[4033.47,4082.25,...|
|2017-08-19 12:00:00|3972.05|[3945.12,4100.0,3...|
|2017-08-19 16:00:00|4076.12|[4036.19,4103.92,...|
|2017-08-19 20:00:00|4139.98|[4

In [31]:
matrix = Correlation.corr(df_assembled, "features")
result = matrix.collect()[0]["pearson(features)"].values
result


array([ 1.        ,  0.99992362,  0.99988658,  0.99985768, -0.18328007,
        0.99992362,  1.        ,  0.99983227,  0.9999224 , -0.18117917,
        0.99988658,  0.99983227,  1.        ,  0.99991324, -0.18624847,
        0.99985768,  0.9999224 ,  0.99991324,  1.        , -0.1835459 ,
       -0.18328007, -0.18117917, -0.18624847, -0.1835459 ,  1.        ])

In [32]:
matrix.collect()[0]["pearson(features)"].values


array([ 1.        ,  0.99992362,  0.99988658,  0.99985768, -0.18328007,
        0.99992362,  1.        ,  0.99983227,  0.9999224 , -0.18117917,
        0.99988658,  0.99983227,  1.        ,  0.99991324, -0.18624847,
        0.99985768,  0.9999224 ,  0.99991324,  1.        , -0.1835459 ,
       -0.18328007, -0.18117917, -0.18624847, -0.1835459 ,  1.        ])

In [33]:
len(features)


5

In [34]:
import numpy as np

correlation_values = matrix.collect()[0]["pearson(features)"].values
num_features = len(features)
correlation_matrix = np.array(correlation_values).reshape(num_features, num_features)


In [35]:
correlation_matrix


array([[ 1.        ,  0.99992362,  0.99988658,  0.99985768, -0.18328007],
       [ 0.99992362,  1.        ,  0.99983227,  0.9999224 , -0.18117917],
       [ 0.99988658,  0.99983227,  1.        ,  0.99991324, -0.18624847],
       [ 0.99985768,  0.9999224 ,  0.99991324,  1.        , -0.1835459 ],
       [-0.18328007, -0.18117917, -0.18624847, -0.1835459 ,  1.        ]])

In [36]:
len(features)


5

In [37]:
# Correlacao direta entre close e volume
df_btc.corr('close', 'volume')


-0.18354590068892643

In [38]:
# Stop the SparkSession
spark.stop()